In [1]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
project = os.getenv('GOOGLE_CLOUD_PROJECT')


# model
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash-lite",
    vertexai = True,
    project = project
)

In [9]:
from langchain.tools import tool

@tool
def say_hello(name:str) -> str:
    """This function is used to greet a person

    Args:
        name (str): name

    Returns:
        str: Greeting
    """
    return f"Hello {name}"


In [10]:
llm_with_tools = llm.bind_tools([say_hello])

In [11]:
# llm which is not aware of tools
response = llm.invoke("Greet shyam")
response

AIMessage(content='Hello Shyam! How are you doing today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d5668-91ca-7d61-a13e-e6ce8a78f8a8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3, 'output_tokens': 9, 'total_tokens': 12, 'input_token_details': {'cache_read': 0}})

In [12]:
response = llm_with_tools.invoke("Greet shyam")
response

AIMessage(content='', additional_kwargs={'function_call': {'name': 'say_hello', 'arguments': '{"name": "shyam"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d5669-3a44-7a43-ba13-eaff3c7a9ecc-0', tool_calls=[{'name': 'say_hello', 'args': {'name': 'shyam'}, 'id': 'a2cb7c78-fdad-4d23-8e04-a29e1316c2e7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 6, 'total_tokens': 42, 'input_token_details': {'cache_read': 0}})

In [13]:
from langchain.agents import create_agent
# create a agent
agent = create_agent(
    model=llm,
    tools=[say_hello]
)

In [14]:
agent_response = agent.invoke({
    "messages": [
        ("user", "Greet shyam")
    ]
})

In [15]:
agent_response['messages'][-1].pretty_print()

================================== Ai Message ==================================

Hello shyam


In [16]:
agent_response['messages']

[HumanMessage(content='Greet shyam', additional_kwargs={}, response_metadata={}, id='60d27960-fe63-47d5-9229-8151df0c914a'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'say_hello', 'arguments': '{"name": "shyam"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d566c-59d1-7801-9dab-bd5b85944c75-0', tool_calls=[{'name': 'say_hello', 'args': {'name': 'shyam'}, 'id': '2ad3c8e0-641e-4a4e-9183-e25124107e46', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 6, 'total_tokens': 42, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='Hello shyam', name='say_hello', id='03bb1290-86f5-49e3-9d4c-28c1e7d689e6', tool_call_id='2ad3c8e0-641e-4a4e-9183-e25124107e46'),
 AIMessage(content='Hello shyam', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 's

In [19]:
## I want to search internet
load_dotenv()
#os.getenv('TAVILY_API_KEY')


True

In [20]:
!uv add langchain-tavily

Resolved 85 packages in 1ms
Audited 81 packages in 58ms


In [43]:
# lets create tavily search tool
from langchain_tavily import TavilySearch

tavily_search_tool = TavilySearch(
    max_results = 3,
    topic = "news"
)

In [44]:
# lets create an agent with tavily search tool

agent = create_agent(
    model=llm,
    tools=[tavily_search_tool]
)

In [47]:
response = agent.invoke({
    "messages": "Get me latest news about gpu's "
})

In [48]:
response['messages']

[HumanMessage(content="Get me latest news about gpu's ", additional_kwargs={}, response_metadata={}, id='e9c8a55e-5850-41d5-b4d7-49a071581f61'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"time_range": "day", "topic": "news", "query": "gpu news"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d5680-5bb3-7ac1-a99c-d8d5566eccfa-0', tool_calls=[{'name': 'tavily_search', 'args': {'time_range': 'day', 'topic': 'news', 'query': 'gpu news'}, 'id': '8f734ec0-4891-4329-b9ef-aa5394a3acff', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1298, 'output_tokens': 14, 'total_tokens': 1312, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='{"query": "gpu news", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://videocardz.com/newz/new-rowhammer-attacks-t

In [49]:
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

Here's the latest news about GPUs:

*   A new Rowhammer attack has been demonstrated on the **GeForce RTX 3060** and an Ampere workstation GPU, with NVIDIA’s earlier security notice explicitly referencing the **RTX A6000** with GDDR6 memory.
*   The **PNY Nvidia GeForce RTX 5070 12GB Graphics Card** is on sale at Walmart for $599. This deal includes a free digital download code of Resident Evil Requiem Standard Edition on Steam while supplies last.
